# `01-scraping.ipynb`: Scraping logs from UMPD website.

## Scraping methodology

UMD's website fortunately uses HTML tables to format its log data, so I will use BeautifulSoup to scrape the tables and load them as pandas dataframes. 

### Scraping methodology for Arrests Log (per year)

### Scraping methodology for Incident and Arrests Logs (per month per year)

In [44]:
# setup 
import os
import requests
import pandas as pd
import geopandas as gpd
import bs4 as bs
import time

HEADERS = {
  "User-Agent": "Jour720/1.0 (diegojto@umd.edu)",
  "Accept-Language": "en-US,en;q=0.9",
}
BASE_ARREST_URL = "https://umpd.umd.edu/statistics-reports/arrest-report-ledgers/"
BASE_CI_URL = "https://umpd.umd.edu/statistics-reports/daily-crime-and-incident-logs/" # CI = Crime and Incident


MONTHS = [f"0{i}" for i in range(1, 10)] + [str(i) for i in range(10, 13)]
CURRENT_YEAR = time.localtime().tm_year
CURRENT_MONTH = time.localtime().tm_mon
ARREST_YEARS = list(range(2018, CURRENT_YEAR + 1))
CI_YEARS = list(range(2019, CURRENT_YEAR + 1))

def text(i, cells):
    return cells[i].get_text(strip=True).strip()

## Scraping Arrests Logs

All of the yearly Arrest Ledger links listed on UMPD's [Arrest Logs](https://umpd.umd.edu/statistics-reports/arrest-logs) website have the following URL structure: `https://umpd.umd.edu/statistics-reports/arrest-report-ledgers/YYYY`, with year options in the range between 2018-2026.

In [ ]:
# Arrest Ledger URL list build
arrest_data = {}
arrest_urls = [f"{BASE_ARREST_URL}{year}" for year in ARREST_YEARS]
start_time = time.time()
for year, url in zip(ARREST_YEARS, arrest_urls):
    year_start_time = time.time()
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()  # Raise an error if the request was unsuccessful
    soup = bs.BeautifulSoup(response.text, "html.parser")
    table = soup.find("table")
    rows = table.find_all("tr")[1:] if table else []
    
    assert rows, f"No rows found for year {year} at URL {url}"
    year_data = []
    for row in rows: 
      cells = row.find_all("td")
      
      if len(cells) < 7:
        print(f"Row has insufficient cells: {len(cells)}")
        continue
      
      year_data.append({
        "arrest_number": text(0, cells),
        "arrest_date": text(1, cells),
        "umpd_case_number": text(2, cells),
        "age": text(3, cells),
        "race": text(4, cells),
        'sex': text(5, cells),
        'arrested_charge': text(6, cells)
      })
    year_end_time = time.time()
    print(f"Scraped {len(year_data)} rows for year {year} in {year_end_time - year_start_time:.2f} seconds.")
    arrest_data[year] = year_data
end_time = time.time()
print(f"Total arrest ledger scraping time: {end_time - start_time:.2f} seconds.")
arrest_dfs = {year: pd.DataFrame(data) for year, data in arrest_data.items()}

Scraped 375 rows for year 2018 in 3.73 seconds.
Scraped 685 rows for year 2019 in 4.55 seconds.
Scraped 324 rows for year 2020 in 1.15 seconds.
Scraped 349 rows for year 2021 in 1.44 seconds.
Scraped 357 rows for year 2022 in 1.92 seconds.
Scraped 300 rows for year 2023 in 1.55 seconds.
Scraped 435 rows for year 2024 in 2.08 seconds.
Scraped 440 rows for year 2025 in 1.83 seconds.
Scraped 463 rows for year 2026 in 6.78 seconds.
Total arrest ledger scraping time: 25.04 seconds.


In [94]:
for year, df in arrest_dfs.items():
  df.to_csv(f"../data/raw/arrest-ledger/arrest_ledger_{year}.csv", index=False)
  
arrest_df_all = pd.concat(arrest_dfs.values(), ignore_index=True)
arrest_df_all.to_csv(f"../data/raw/arrest-ledger/umpd_arrest_ledger_2018-{CURRENT_YEAR}.csv", index=False)

# Scraping Incident and Arrests Logs

The [Daily Crime and Incident Logs](https://umpd.umd.edu/statistics-reports/daily-crime-and-incident-logs) have the following URL structure: `https://umpd.umd.edu/statistics-reports/daily-crime-and-incident-logs/YYYY/MM`, year options being (2019-2026) and month options being (01-12), except 2026 will have to have current month.

**NOTE**: The website as of 2026-11-06 has not been updated to include November Daily logs. This code is written to scrape every crime and incident log to the month prior of running this script. 

In [49]:
ci_data = {year: {int(month): [] for month in MONTHS} for year in CI_YEARS}
ci_urls = [f"{BASE_CI_URL}{year}/{month}" for year in CI_YEARS for month in MONTHS if not (year == CURRENT_YEAR and int(month) >= CURRENT_MONTH)]
for url in ci_urls: 
  url_parts = url.split("/")
  year, month = url_parts[-2], url_parts[-1]
  response = requests.get(url, headers=HEADERS, timeout=10)
  response.raise_for_status() 
  soup = bs.BeautifulSoup(response.text, "html.parser")
  table = soup.find("table")
  rows = table.find_all("tr")[1:] if table else []
  
  assert rows, f"No rows found for year {year}, month {month} at URL {url}"
  
  for i in range(len(rows)-1):
    row = rows[i]
    row_next = rows[i+1]
    
    row_cells = row.find_all("td")
    row_next_cells = row_next.find_all("td")
    
    if len(row_cells) == 1 and len(row_next_cells) != 1:
      continue
    
    # if (len(row_cells) < 5 or len(row_next_cells) != 1) and (len(row_cells) + len(row_next_cells) != 6):
    #   print(f"Row {i} or Row {i+1} has insufficient cells: {len(row_cells)} and {len(row_next_cells)}")
    #   continue
    
    ci_data[int(year)][int(month)].append({
      'umpd_case_number': text(0, row_cells),
      'date_occurred': text(1, row_cells),
      'report_date': text(2, row_cells),
      'crime_type': text(3, row_cells),
      'disposition': text(4, row_cells),
      'location': text(0, row_next_cells)
    })


In [ ]:
ci_dfs_individual = {year: [] for year in CI_YEARS}
ci_dfs_combined_yearly = {year: None for year in CI_YEARS}
for year, months in ci_data.items():
  os.makedirs(f"../data/raw/daily-crime-incident-logs/{year}", exist_ok=True)
  month_list = []
  for month, data in months.items():
    df = pd.DataFrame(data)
    if df.empty:
      print(f"No data for year {year}, month {month}. Skipping CSV creation.")
      continue
    month_list.append(df)
    df.to_csv(f"../data/raw/daily-crime-incident-logs/{year}/daily_crime_incident_log_{year}_{month}.csv", index=False)
  ci_dfs_individual[year] = month_list
  ci_dfs_combined_yearly[year] = pd.concat(month_list, ignore_index=True)
  ci_dfs_combined_yearly[year].to_csv(f"../data/raw/daily-crime-incident-logs/daily_crime_incident_log_{year}.csv", index=False)

ci_df_all = pd.concat([df for df in ci_dfs_combined_yearly.values() if not df.empty], ignore_index=True)
ci_df_all.to_csv(f"../data/raw/daily-crime-incident-logs/daily_crime_incident_log_2019-{CURRENT_YEAR}.csv", index=False)

No data for year 2026, month 11. Skipping CSV creation.
No data for year 2026, month 12. Skipping CSV creation.


## Combining `ci_df_all` and `arrests_df_all` where they match

In [103]:
merge_results = pd.merge(arrest_df_all, ci_df_all, on="umpd_case_number", how="inner")

merge_results['umpd_case_number'].value_counts().sort_values(ascending=False)

umpd_case_number
2019-00025263    10
2021-00081637    10
2025-00088127     8
2024-00118283     6
2024-00177296     6
                 ..
2021-00035276     1
2021-00035475     1
2021-00035772     1
2021-00024023     1
2026-00289157     1
Name: count, Length: 2944, dtype: int64

In [104]:
merge_results.to_csv(f"../data/processed/merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv", index=False)